# 標記樣本拍攝（在 JetBot 上跑）

拍校正用的照片。**兩種照片要分開拍，用途不一樣：**

| 檔名前綴 | 距離 | 用途 |
|---|---|---|
| `close` | 約 0.3 m 特寫 | **量顏色**。貼紙要夠大才框得準 |
| `0.5m` `0.8m` `1.5m` `2.0m` | 站立全身 | **驗證偵測距離**與像素大小 |

---

### 拍之前先確認三件事

1. **攝影機角度已經固定**。角度一改，所有數字都要重來
2. **解析度是 640×480（4:3）** —— 感測器原生比例，不會被壓扁。
   下一格的註解有說明它與上車推論解析度不一致的影響
3. **存 PNG，不要存 JPEG**。JPEG 的色度抽樣（4:2:0）會把色彩解析度砍半，
   對只有幾個像素的色塊是災難性的。下面的程式已經寫死 PNG

拍完把整個 `samples/` 資料夾複製回筆電，跑 `analyze_marker.py`。

In [ ]:
import os

import cv2
import traitlets
import ipywidgets as widgets
from IPython.display import display
from jetbot import Camera, bgr8_to_jpeg

SAVE_DIR = 'samples'
if not os.path.isdir(SAVE_DIR):
    os.makedirs(SAVE_DIR)

# 640x480(4:3)。這是感測器的原生比例(816x616),不會被壓扁變形。
#
# ⚠️ 這個解析度與「上車推論時的畫面大小」目前**不一致**:
#      校正拍照   640x480  = 307,200 px
#      上車推論   300x300  =  90,000 px   (live_demo_following_sm.ipynb)
#
#    影響分成兩半:
#      HSV 顏色門檻(H/S/V 範圍)  → **與解析度無關**,在哪拍都一樣
#      面積門檻(min_pink_area)   → **與解析度有關**,640x480 量到的數字
#                                   直接套到 300x300 會太大、抓不到標記
#
#    所以 analyze_marker.py 的兩個模式用途不同:
#      --pick   量顏色    → 這個解析度拍沒問題,畫面大反而好框
#      --check  驗證面積  → 數字只在同解析度下成立,見該腳本輸出的提示
#
#    若日後把上車的相機也改成 640x480,兩邊就一致,這段註解可以刪掉。
CAM_W, CAM_H = 640, 480

camera = Camera.instance(width=CAM_W, height=CAM_H)
print('相機解析度:%dx%d' % (CAM_W, CAM_H))

# 顯示縮小一半,免得把瀏覽器畫面塞爆;存檔仍是 640x480 原圖
image_widget = widgets.Image(format='jpeg', width=CAM_W // 2, height=CAM_H // 2)
camera_link = traitlets.dlink((camera, 'value'), (image_widget, 'value'),
                              transform=bgr8_to_jpeg)
display(image_widget)

### 拍攝清單

每個情境改一次「檔名前綴」再按拍照，檔名會自動編號。

**量顏色（前綴 `close`）—— 5~8 張就夠**
- 一般室內燈光下 2 張
- 靠窗、光線比較亮 2 張
- 陰影裡、光線比較暗 2 張
- 貼紙稍微傾斜（模擬走路時腿的角度）2 張

光線變化一定要涵蓋，不然門檻會只在拍攝當下那個亮度成立。

**驗證距離（前綴 `0.5m` / `0.8m` / `1.5m` / `2.0m`）—— 每個距離 3~4 張**
- 目標者站定、正面 1 張
- 側身 1 張
- 走路中途（兩腿交錯的那一瞬間）1~2 張 ← **最重要，這是最容易失敗的狀況**

**如果霧面和亮面兩種材質都買了**，各拍一輪，前綴加上 `matte` / `gloss`
（例如 `gloss_close`、`matte_1.5m`），分開跑 `--pick` 就能直接比過曝比例。

In [ ]:
prefix_widget = widgets.Text(value='close', description='檔名前綴')
count_widget = widgets.IntText(value=0, description='資料夾張數', disabled=True)
snap_button = widgets.Button(description='拍一張', button_style='success')
undo_button = widgets.Button(description='刪掉上一張', button_style='danger')

_last = {'path': None}


def _refresh_count():
    count_widget.value = len([f for f in os.listdir(SAVE_DIR)
                              if f.lower().endswith('.png')])


def save_one(_):
    prefix = prefix_widget.value.strip() or 'sample'
    n = 1
    while True:
        path = os.path.join(SAVE_DIR, '%s_%02d.png' % (prefix, n))
        if not os.path.exists(path):
            break
        n += 1
    # camera.value 就是 BGR8 的 numpy 陣列，imwrite 直接吃，不要先轉色彩空間
    cv2.imwrite(path, camera.value)
    _last['path'] = path
    _refresh_count()
    print('存了 %s' % path)


def undo_one(_):
    path = _last['path']
    if path and os.path.exists(path):
        os.remove(path)
        _last['path'] = None
        _refresh_count()
        print('刪掉 %s' % path)
    else:
        print('沒有可刪的（只能刪剛拍的那一張）')


snap_button.on_click(save_one)
undo_button.on_click(undo_one)
_refresh_count()

display(widgets.VBox([
    prefix_widget,
    widgets.HBox([snap_button, undo_button]),
    count_widget,
]))

### 收尾

拍完一定要跑下面這格把相機釋放掉，否則其他 notebook（包括 `live_demo_following_sm.ipynb`）
會開不起來，錯誤訊息通常看不出是這個原因。

萬一忘了跑就關掉 notebook，到終端機執行 Kernel → Restart，或重開 JetBot 的 Jupyter。

In [ ]:
camera_link.unlink()
camera.stop()
print('相機已釋放。要繼續拍請從第一格重跑。')